# Iterative Taylor Pruning Visualization: HML Benchmarks

Visualize iterative Taylor head pruning results across:
- 15 models (requested DeepSeek/Qwen/Llama/GPT-OSS/Gemma/Phi suite)
- 3 datasets: HumanEval, MBPP, LiveCodeBench
- 2 modes: Regular (CoT), Chain_Code

Each dataset+mode is pruned independently. Compare convergence, head distributions, and final losses.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
%matplotlib inline

BASE_DIR = Path('../../results/hml')
REQUESTED_MODELS = [
    'Qwen--Qwen3-0.6B',
    'Qwen--Qwen3-4B',
    'Qwen--Qwen3-8B',
    'Qwen--Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b',
    'google--gemma-4-E2B-it',
    'google--gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning',
]
MODEL_DIR_BY_REQUEST = {
    'Qwen--Qwen3-0.6B': 'Qwen3-0.6B',
    'Qwen--Qwen3-4B': 'Qwen3-4B',
    'Qwen--Qwen3-8B': 'Qwen3-8B',
    'Qwen--Qwen3-14B': 'Qwen3-14B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B': 'DeepSeek-R1-Distill-Qwen-1.5B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-7B': 'DeepSeek-R1-Distill-Qwen-7B',
    'deepseek-ai--DeepSeek-R1-Distill-Llama-8B': 'DeepSeek-R1-Distill-Llama-8B',
    'deepseek-ai--DeepSeek-R1-Distill-Qwen-14B': 'DeepSeek-R1-Distill-Qwen-14B',
    'deepseek-ai--DeepSeek-R1-0528-Qwen3-8B': 'DeepSeek-R1-0528-Qwen3-8B',
    'meta-llama--Llama-3.2-3B-Instruct': 'Llama-3.2-3B-Instruct',
    'openai--gpt-oss-20b': 'gpt-oss-20b',
    'google--gemma-4-E2B-it': 'gemma-4-E2B-it',
    'google--gemma-4-E4B-it': 'gemma-4-E4B-it',
    'google--gemma-4-26B-A4B-it': 'gemma-4-26B-A4B-it',
    'microsoft--Phi-4-mini-reasoning': 'Phi-4-mini-reasoning',
}
MODEL_NAMES = [MODEL_DIR_BY_REQUEST[m] for m in REQUESTED_MODELS]
MODEL_DISPLAY = {MODEL_DIR_BY_REQUEST[m]: m for m in REQUESTED_MODELS}
MODEL_SHORT = {
    'Qwen3-0.6B': 'Q3-0.6B',
    'Qwen3-4B': 'Q3-4B',
    'Qwen3-8B': 'Q3-8B',
    'Qwen3-14B': 'Q3-14B',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'DS-DQ1.5B',
    'DeepSeek-R1-Distill-Qwen-7B': 'DS-DQ7B',
    'DeepSeek-R1-Distill-Llama-8B': 'DS-DL8B',
    'DeepSeek-R1-Distill-Qwen-14B': 'DS-DQ14B',
    'DeepSeek-R1-0528-Qwen3-8B': 'DS-0528-Q8B',
    'Llama-3.2-3B-Instruct': 'Llama3.2-3B',
    'gpt-oss-20b': 'gpt-oss-20b',
    'gemma-4-E2B-it': 'Gemma4-E2B',
    'gemma-4-E4B-it': 'Gemma4-E4B',
    'gemma-4-26B-A4B-it': 'Gemma4-26B',
    'Phi-4-mini-reasoning': 'Phi4-mini',
}
DATASETS = ['humaneval', 'mbpp', 'livecodebench']
MODES = {'regular': 'iterative_taylor', 'chain_code': 'iterative_taylor_chain_code'}

palette = sns.color_palette('tab20', n_colors=max(12, len(MODEL_NAMES)))
MODEL_COLORS = {model: palette[i] for i, model in enumerate(MODEL_NAMES)}
DS_COLORS = {'humaneval': 'tab:green', 'mbpp': 'tab:purple', 'livecodebench': 'tab:red'}
MODE_COLORS = {'regular': 'tab:blue', 'chain_code': 'tab:red'}
MODE_STYLES = {'regular': '-', 'chain_code': '--'}

def model_label(model, short=False):
    if short:
        return MODEL_SHORT.get(model, model)
    return MODEL_DISPLAY.get(model, model)


def grid_shape(n_items, max_cols=4):
    cols = min(max_cols, n_items)
    rows = int(np.ceil(n_items / cols))
    return rows, cols


def symmetric_percentile_limits(arrays, low=1, high=99):
    values = []
    for arr in arrays:
        if arr is None:
            continue
        arr = np.asarray(arr)
        finite = arr[np.isfinite(arr)]
        if finite.size == 0:
            continue
        values.append(np.percentile(finite, low))
        values.append(np.percentile(finite, high))

    if not values:
        return -1.0, 1.0

    max_abs_val = max(abs(min(values)), abs(max(values)))
    if not np.isfinite(max_abs_val) or max_abs_val == 0:
        max_abs_val = 1.0
    return -max_abs_val, max_abs_val


def get_plot_losses(data):
    history = data.get('history', [])
    iterations = [int(h.get('iteration', i)) for i, h in enumerate(history)]
    losses = [float(h.get('loss', np.nan)) for h in history]
    return iterations, losses


## Load Results

In [ ]:
def load_taylor_result(ds, mode_dir, model):
    """Load iterative taylor results from json or npz."""
    json_path = BASE_DIR / ds / mode_dir / model / 'iterative_taylor_results.json'
    npz_path = BASE_DIR / ds / mode_dir / model / 'iterative_taylor_results.npz'
    if json_path.exists():
        with open(json_path) as f:
            return json.load(f)
    elif npz_path.exists():
        npz = np.load(npz_path)
        return {
            'baseline_loss': float(npz['baseline_loss'][0]),
            'history': [{'iteration': int(i), 'alive_heads': int(a), 'loss': float(l)}
                        for i, a, l in zip(npz['iterations'], npz['alive_counts'], npz['losses'])],
            'alive_heads': npz['alive_heads'].tolist(),
        }
    return None


def cap_result_to_model_quarter(data):
    """Keep points up to removing <= 1/4 heads, and align alive_heads to that checkpoint."""
    history = data.get('history', [])
    if not history:
        return data

    total_heads = int(history[0]['alive_heads'])
    max_removed = total_heads / 2.0

    # Pick the closest available checkpoint that does not exceed 50% removal.
    cap_idx = 0
    for idx, h in enumerate(history):
        removed = total_heads - int(h['alive_heads'])
        if removed <= max_removed:
            cap_idx = idx
        else:
            break

    capped = dict(data)
    capped['history'] = history[:cap_idx + 1]

    # Reconstruct surviving heads at the capped checkpoint when pruning steps are available.
    try:
        num_layers = int(data.get('num_layers'))
        num_heads = int(data.get('num_heads'))
    except (TypeError, ValueError):
        num_layers, num_heads = None, None

    if num_layers is not None and num_heads is not None:
        alive_set = {(l, h) for l in range(num_layers) for h in range(num_heads)}
        for step in history[1:cap_idx + 1]:
            for layer, head in step.get('pruned_this_iter', []):
                alive_set.discard((int(layer), int(head)))

        capped_alive = sorted(alive_set)
        expected_alive = int(capped['history'][-1]['alive_heads'])
        capped['alive_heads'] = capped_alive if len(capped_alive) == expected_alive else data.get('alive_heads', [])
    else:
        capped['alive_heads'] = data.get('alive_heads', [])

    capped['cap_total_heads'] = total_heads
    capped['cap_removed_target'] = max_removed
    capped['cap_removed'] = total_heads - int(capped['history'][-1]['alive_heads'])
    return capped


# all_results[mode][model][dataset]
all_results = {}
for mode, mode_dir in MODES.items():
    all_results[mode] = {}
    for model in MODEL_NAMES:
        all_results[mode][model] = {}
        print(f'=== {mode} | {model} ===')
        for ds in DATASETS:
            data = load_taylor_result(ds, mode_dir, model)
            if data:
                capped_data = cap_result_to_model_quarter(data)
                all_results[mode][model][ds] = capped_data
                n_iters = len(capped_data['history']) - 1
                final = capped_data['history'][-1]['loss']
                removed = capped_data['cap_removed']
                total = capped_data['cap_total_heads']
                print(
                    f'  {ds}: {n_iters} iters, baseline={capped_data["baseline_loss"]:.4f}, '
                    f'final={final:.4f}, removed={removed}/{total}'
                )
            else:
                print(f'  {ds}: NOT FOUND')
        print()

print('Using per-model cap: closest checkpoint with <=25% heads removed')
for model in MODEL_NAMES:
    cap_removed_vals = []
    total_heads_vals = []
    for mode in MODES:
        for ds in DATASETS:
            data = all_results.get(mode, {}).get(model, {}).get(ds)
            if data is None:
                continue
            cap_removed_vals.append(int(data['cap_removed']))
            total_heads_vals.append(int(data['cap_total_heads']))

    if not cap_removed_vals:
        continue

    total_heads = total_heads_vals[0]
    target = total_heads / 4.0
    if min(cap_removed_vals) == max(cap_removed_vals):
        cap_removed = cap_removed_vals[0]
        print(
            f'  {model}: cap_removed={cap_removed} (target<={target:.2f}), '
            f'cap_alive={total_heads - cap_removed}'
        )
    else:
        print(
            f'  {model}: cap_removed range={min(cap_removed_vals)}-{max(cap_removed_vals)} '
            f'(target<={target:.2f})'
        )

## 1. Pruning Convergence Curves

In [ ]:
# Per-mode per-model: datasets overlaid
for mode in MODES:
    rows, cols = grid_shape(len(MODEL_NAMES), max_cols=4)
    fig, axes = plt.subplots(rows, cols, figsize=(6.5 * cols, 4.8 * rows), squeeze=False)
    for m_idx, model in enumerate(MODEL_NAMES):
        r, c = divmod(m_idx, cols)
        ax = axes[r, c]
        for ds in DATASETS:
            data = all_results[mode][model].get(ds)
            if data is None:
                continue
            iters, losses = get_plot_losses(data)
            if len(losses) == 0:
                continue
            label = f'{ds} (final: {losses[-1]:.4f})'
            ax.plot(iters, losses, linewidth=2, label=label, color=DS_COLORS[ds])
        ax.set_xlabel('Pruning Iteration', fontsize=11)
        ax.set_ylabel('Loss', fontsize=11)
        ax.set_title(model_label(model, short=True), fontsize=12)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    total_slots = rows * cols
    for empty_idx in range(len(MODEL_NAMES), total_slots):
        r, c = divmod(empty_idx, cols)
        axes[r, c].axis('off')

    fig.suptitle(f'Loss vs Iteration ({mode})', fontsize=15, y=1.01)
    fig.tight_layout()
    plt.show()


In [ ]:
# Per-dataset: regular vs chain_code for each model
for model in MODEL_NAMES:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(8 * len(DATASETS), 6), squeeze=False)
    for d_idx, ds in enumerate(DATASETS):
        ax = axes[0, d_idx]
        for mode in MODES:
            data = all_results[mode][model].get(ds)
            if data is None:
                continue
            iters, losses = get_plot_losses(data)
            if len(losses) == 0:
                continue

            label = f'{mode} (final: {losses[-1]:.4f})'
            ax.plot(
                iters,
                losses,
                MODE_STYLES[mode],
                color=MODE_COLORS[mode],
                linewidth=2,
                label=label,
            )
        ax.set_xlabel('Pruning Iteration', fontsize=12)
        ax.set_ylabel('Loss', fontsize=12)
        ax.set_title(f'{ds}', fontsize=14)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f'{model} - Regular vs Chain_Code', fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()


## 2. Loss vs Heads Remaining

In [ ]:
for mode in MODES:
    rows, cols = grid_shape(len(MODEL_NAMES), max_cols=4)
    fig, axes = plt.subplots(rows, cols, figsize=(6.5 * cols, 4.8 * rows), squeeze=False)
    for m_idx, model in enumerate(MODEL_NAMES):
        r, c = divmod(m_idx, cols)
        ax = axes[r, c]
        for ds in DATASETS:
            data = all_results[mode][model].get(ds)
            if data is None:
                continue
            alive = [h['alive_heads'] for h in data['history']]
            _, losses = get_plot_losses(data)
            if len(losses) == 0:
                continue
            if len(alive) != len(losses):
                n = min(len(alive), len(losses))
                alive = alive[:n]
                losses = losses[:n]
            ax.plot(alive, losses, linewidth=2, label=ds, color=DS_COLORS[ds], marker='o', markersize=2)
        ax.set_xlabel('Alive Heads', fontsize=11)
        ax.set_ylabel('Loss', fontsize=11)
        ax.set_title(model_label(model, short=True), fontsize=12)
        ax.legend(fontsize=8)
        ax.invert_xaxis()
        ax.grid(True, alpha=0.3)

    total_slots = rows * cols
    for empty_idx in range(len(MODEL_NAMES), total_slots):
        r, c = divmod(empty_idx, cols)
        axes[r, c].axis('off')

    fig.suptitle(f'Loss vs Remaining Heads ({mode})', fontsize=15, y=1.01)
    fig.tight_layout()
    plt.show()

## 3. Surviving Head Distribution

In [ ]:
for model in MODEL_NAMES:
    # Find datasets that exist for this model in at least one mode
    dsets = [ds for ds in DATASETS if any(ds in all_results[m][model] and 'alive_heads' in all_results[m][model][ds] for m in MODES)]
    
    if not dsets:
        continue
        
    fig, axes = plt.subplots(1, len(dsets), figsize=(7*len(dsets), 5), squeeze=False)
    
    for idx, ds in enumerate(dsets):
        ax = axes[0, idx]
        
        layers_data = []
        labels = []
        colors = []
        num_layers = 0
        
        # Gather data for both modes for the current dataset
        for mode in MODES:
            if ds in all_results[mode][model] and 'alive_heads' in all_results[mode][model][ds]:
                alive = np.array(all_results[mode][model][ds]['alive_heads'])
                if len(alive) > 0:
                    layers = alive[:, 0]
                    layers_data.append(layers)
                    
                    # Add total heads to the legend label
                    labels.append(f'{mode} ({len(alive)} heads)') 
                    colors.append(MODE_COLORS[mode]) # Uses your existing mode colors
                    
                    # Keep track of the maximum layer count across modes
                    num_layers = max(num_layers, int(layers.max()) + 1)
        
        if not layers_data:
            continue
            
        # Passing a list of arrays to ax.hist automatically groups them side-by-side
        ax.hist(layers_data, bins=num_layers, range=(-0.5, num_layers-0.5),
                color=colors, label=labels, alpha=0.7, edgecolor='black')
                
        ax.set_xlabel('Layer')
        ax.set_ylabel('Surviving Heads')
        ax.set_title(f'{ds}', fontsize=13)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        
    fig.suptitle(f'{model} - Surviving Head Distribution (Mode Comparison)', fontsize=15, y=1.02)
    fig.tight_layout()
    plt.show()

## 4. Heatmap of Surviving Heads

In [ ]:
for mode in MODES:
    for model in MODEL_NAMES:
        dsets = [ds for ds in DATASETS if ds in all_results[mode][model]
                 and 'alive_heads' in all_results[mode][model][ds]]
        if not dsets:
            continue

        matrices = []
        for ds in dsets:
            alive = np.array(all_results[mode][model][ds]['alive_heads'])
            if len(alive) == 0:
                continue
            nl = int(alive[:, 0].max()) + 1
            nh = int(alive[:, 1].max()) + 1
            matrix = np.zeros((nl, nh))
            for layer, head in alive:
                matrix[layer, head] = 1
            matrices.append(matrix)

        symmetric_vmin, symmetric_vmax = symmetric_percentile_limits(matrices)

        fig, axes = plt.subplots(1, len(dsets), figsize=(8 * len(dsets), 8), squeeze=False)
        mappable = None
        for idx, ds in enumerate(dsets):
            alive = np.array(all_results[mode][model][ds]['alive_heads'])
            if len(alive) == 0:
                continue
            nl = int(alive[:, 0].max()) + 1
            nh = int(alive[:, 1].max()) + 1
            matrix = np.zeros((nl, nh))
            for layer, head in alive:
                matrix[layer, head] = 1
            ax = axes[0, idx]
            mappable = ax.imshow(
                matrix,
                aspect='auto',
                cmap='coolwarm',
                interpolation='nearest',
                vmin=symmetric_vmin,
                vmax=symmetric_vmax,
            )
            ax.set_xlabel('Head'); ax.set_ylabel('Layer')
            ax.set_title(f'{ds}', fontsize=13)

        fig.subplots_adjust(right=0.9)
        cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        if mappable is not None:
            fig.colorbar(mappable, cax=cbar_ax, label='Alive')
        fig.suptitle(f'{model} ({mode}) - Surviving Heads', fontsize=15, y=1.02)
        plt.show()

## 5. Cross-Dataset Head Overlap

In [ ]:
for mode in MODES:
    for model in MODEL_NAMES:
        dsets = [ds for ds in DATASETS if ds in all_results[mode][model]
                 and 'alive_heads' in all_results[mode][model][ds]]
        if len(dsets) < 2:
            continue
            
        head_sets = {ds: set(map(tuple, all_results[mode][model][ds]['alive_heads'])) for ds in dsets}
        n = len(dsets)
        jaccard = np.zeros((n, n))
        overlap_counts = np.zeros((n, n), dtype=int)
        
        for i, di in enumerate(dsets):
            for j, dj in enumerate(dsets):
                inter = len(head_sets[di] & head_sets[dj])
                union = len(head_sets[di] | head_sets[dj])
                jaccard[i, j] = inter / union if union > 0 else 0
                overlap_counts[i, j] = inter

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # --- Plot 1: Overlap Counts ---
        im1 = ax1.imshow(
            overlap_counts,
            cmap='coolwarm',
            interpolation='nearest',
            vmin=0, 
            vmax=overlap_counts.max() # Scale specific to count data
        )
        ax1.set_xticks(range(n)); ax1.set_yticks(range(n))
        ax1.set_xticklabels(dsets); ax1.set_yticklabels(dsets)
        ax1.set_title(f'{model} ({mode}) - Overlap')
        fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04) # Dedicated colorbar
        
        for i in range(n):
            for j in range(n):
                ax1.text(j, i, str(overlap_counts[i, j]), ha='center', va='center', fontsize=12, fontweight='bold')

        # --- Plot 2: Jaccard Similarity ---
        im2 = ax2.imshow(
            jaccard,
            cmap='coolwarm',
            interpolation='nearest',
            vmin=0.0, 
            vmax=1.0 # Jaccard is always between 0 and 1
        )
        ax2.set_xticks(range(n)); ax2.set_yticks(range(n))
        ax2.set_xticklabels(dsets); ax2.set_yticklabels(dsets)
        ax2.set_title(f'{model} ({mode}) - Jaccard')
        fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04) # Dedicated colorbar
        
        for i in range(n):
            for j in range(n):
                ax2.text(j, i, f'{jaccard[i, j]:.2f}', ha='center', va='center', fontsize=12, fontweight='bold')

        fig.tight_layout()
        plt.show()
        
        common = set.intersection(*head_sets.values())
        print(f'{model} ({mode}): {len(common)} heads common to all datasets')

In [ ]:
print("--- Common Heads Between Modes ---")

for model in MODEL_NAMES:
    for ds in DATASETS:
        heads_per_mode = []
        modes_found = []
        
        for mode in MODES:
            # Safely check if the nested dictionaries exist before accessing
            if (mode in all_results and 
                model in all_results[mode] and 
                ds in all_results[mode][model] and 
                'alive_heads' in all_results[mode][model][ds]):
                
                # Convert the list of heads to a set of tuples for easy intersection
                heads = set(map(tuple, all_results[mode][model][ds]['alive_heads']))
                heads_per_mode.append(heads)
                modes_found.append(mode)
        
        # We need at least 2 modes to do a meaningful intersection
        if len(heads_per_mode) > 1:
            common_heads = set.intersection(*heads_per_mode)
            print(f"{model} | {ds}: {len(common_heads)} common heads (across {', '.join(modes_found)})")
        elif len(heads_per_mode) == 1:
            print(f"{model} | {ds}: Only 1 mode found ({modes_found[0]}), cannot compare.")
        else:
            pass # Skip printing if no data exists at all for this model/dataset

## 6. Regular vs Chain_Code: Surviving Head Overlap

In [ ]:
# For each model+dataset, compare surviving heads between the two modes
print(f'{"Model":>25} | {"Dataset":>13} | {"Reg alive":>10} | {"CC alive":>10} | {"Overlap":>8} | {"Jaccard":>8}')
print('-'*85)

for model in MODEL_NAMES:
    for ds in DATASETS:
        reg = all_results['regular'][model].get(ds)
        cc = all_results['chain_code'][model].get(ds)
        if not (reg and cc and 'alive_heads' in reg and 'alive_heads' in cc):
            continue
        reg_set = set(map(tuple, reg['alive_heads']))
        cc_set = set(map(tuple, cc['alive_heads']))
        inter = len(reg_set & cc_set)
        union = len(reg_set | cc_set)
        jac = inter / union if union > 0 else 0
        print(f'{model:>25} | {ds:>13} | {len(reg_set):>10} | {len(cc_set):>10} | {inter:>8} | {jac:8.3f}')

## 7. Cross-Reference: Ablation Importance vs Taylor Pruning

In [ ]:
TOP_K = 20

for mode, mode_dir in MODES.items():
    abl_subdir = 'head_ablation_results' if mode == 'regular' else 'head_ablation_results_chain_code'
    for model in MODEL_NAMES:
        print(f'\n{"="*60}')
        print(f'  {model} ({mode})')
        print(f'{"="*60}')
        for ds in DATASETS:
            data = all_results[mode][model].get(ds)
            if not data or 'alive_heads' not in data:
                continue
            alive_set = set(map(tuple, data['alive_heads']))
            ablation_path = BASE_DIR / ds / abl_subdir / model / ds / 'ablation.npz'
            if not ablation_path.exists():
                print(f'  {ds}: ablation data not found')
                continue
            abl = np.load(ablation_path)['ablation_mean']
            flat = abl.flatten()
            top_idx = np.argsort(flat)[-TOP_K:][::-1]
            num_heads = abl.shape[1]
            survived = sum(1 for idx in top_idx if (idx // num_heads, idx % num_heads) in alive_set)
            print(f'  {ds}: {survived}/{TOP_K} top ablation heads survived Taylor pruning')

## 8. Summary

In [ ]:
summary_data = []
for mode in MODES:
    for model in MODEL_NAMES:
        for ds in DATASETS:
            data = all_results[mode][model].get(ds)
            if data is None:
                continue
            history = data['history']
            baseline = data['baseline_loss']
            final = history[-1]['loss']
            n_alive = history[-1]['alive_heads']
            total = history[0]['alive_heads']
            summary_data.append({
                'Mode': mode, 'Model': model, 'Dataset': ds,
                'Baseline': f'{baseline:.4f}',
                'Final': f'{final:.4f}',
                'Delta': f'{final - baseline:+.4f}',
                'Alive/Total': f'{n_alive}/{total}',
                'Pruned %': f'{100*(total-n_alive)/total:.1f}%',
                'Iters': len(history)-1,
            })

df = pd.DataFrame(summary_data)
print('\n' + '='*105)
print('SUMMARY - ITERATIVE TAYLOR PRUNING (HML)')
print('='*105)
print(df.to_string(index=False))
print('='*105)

## 9. Interactive Iteration Explorer (Clean)

Use dropdowns for model, dataset, and mode, then scrub pruning iterations with a slider. This version avoids ad-hoc fallbacks and keeps all selection logic explicit.

In [ ]:
import json

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.colors import BoundaryNorm, ListedColormap

RESULT_FILE = "iterative_taylor_results.json"

if not MODEL_NAMES:
    raise ValueError("MODEL_NAMES is empty. Run the setup cells first.")
if not DATASETS:
    raise ValueError("DATASETS is empty. Run the setup cells first.")
if not MODES:
    raise ValueError("MODES is empty. Run the setup cells first.")

HEAD_CMAP = ListedColormap(["#d9d9d9", "#8b0000"])
HEAD_NORM = BoundaryNorm([-0.5, 0.5, 1.5], HEAD_CMAP.N)

def _infer_shape_from_history(history, num_layers, num_heads):
    if num_layers > 0 and num_heads > 0:
        return num_layers, num_heads

    pruned_coords = [
        (int(layer), int(head))
        for step in history
        for layer, head in step.get("pruned_this_iter", [])
    ]
    max_layer = max([layer for layer, _ in pruned_coords], default=-1)
    max_head = max([head for _, head in pruned_coords], default=-1)

    inferred_layers = max(num_layers, max_layer + 1)
    inferred_heads = max(num_heads, max_head + 1)
    if inferred_layers <= 0 or inferred_heads <= 0:
        raise ValueError("Could not infer num_layers/num_heads from result history.")

    return inferred_layers, inferred_heads


def load_iteration_view_data(model_name, dataset, mode):
    mode_dir = MODES[mode]
    json_path = BASE_DIR / dataset / mode_dir / model_name / RESULT_FILE

    if not json_path.exists():
        raise FileNotFoundError(f"Missing result file: {json_path}")

    with open(json_path, "r") as f:
        taylor_data = json.load(f)

    history = taylor_data.get("history", [])
    if not history:
        raise ValueError(f"No history entries found in: {json_path}")

    num_layers = int(taylor_data.get("num_layers", 0) or 0)
    num_heads = int(taylor_data.get("num_heads", 0) or 0)
    num_layers, num_heads = _infer_shape_from_history(history, num_layers, num_heads)

    iterations = np.array([int(step.get("iteration", i)) for i, step in enumerate(history)], dtype=int)
    losses = np.array([float(step.get("loss", np.nan)) for step in history], dtype=float)

    return {
        "model_name": model_name,
        "dataset": dataset,
        "mode": mode,
        "json_path": json_path,
        "history": history,
        "iterations": iterations,
        "losses": losses,
        "num_layers": int(num_layers),
        "num_heads": int(num_heads),
    }


def build_alive_mask(payload, iteration_idx):
    mask = np.ones((payload["num_layers"], payload["num_heads"]), dtype=np.int8)
    for step in payload["history"][1 : iteration_idx + 1]:
        for layer, head in step.get("pruned_this_iter", []):
            layer = int(layer)
            head = int(head)
            if 0 <= layer < payload["num_layers"] and 0 <= head < payload["num_heads"]:
                mask[layer, head] = 0
    return mask


def plot_iteration(payload, iteration_idx):
    fig, (ax_loss, ax_heat) = plt.subplots(1, 2, figsize=(14, 5))

    iterations = payload["iterations"]
    losses = payload["losses"]

    ax_loss.plot(iterations, losses, color="#4b5563", linewidth=2, label="loss")
    ax_loss.scatter(
        [iterations[iteration_idx]],
        [losses[iteration_idx]],
        s=90,
        color="red",
        edgecolors="black",
        linewidths=0.8,
        zorder=3,
    )
    ax_loss.set_title("Loss Trajectory")
    ax_loss.set_xlabel("Iteration")
    ax_loss.set_ylabel("Loss")
    ax_loss.grid(True, alpha=0.3)

    alive_mask = build_alive_mask(payload, iteration_idx)
    im = ax_heat.imshow(
        alive_mask,
        cmap=HEAD_CMAP,
        norm=HEAD_NORM,
        interpolation="nearest",
        aspect="auto",
    )
    ax_heat.set_title(f"Head State at Iteration {int(iterations[iteration_idx])}")
    ax_heat.set_xlabel("Head")
    ax_heat.set_ylabel("Layer")

    x_step = max(1, payload["num_heads"] // 16)
    y_step = max(1, payload["num_layers"] // 16)
    ax_heat.set_xticks(np.arange(0, payload["num_heads"], x_step))
    ax_heat.set_yticks(np.arange(0, payload["num_layers"], y_step))
    ax_heat.set_xticks(np.arange(-0.5, payload["num_heads"], 1), minor=True)
    ax_heat.set_yticks(np.arange(-0.5, payload["num_layers"], 1), minor=True)
    ax_heat.grid(which="minor", color="white", linestyle="-", linewidth=0.6)
    ax_heat.tick_params(which="minor", bottom=False, left=False)

    cbar = fig.colorbar(im, ax=ax_heat, ticks=[0, 1], fraction=0.046, pad=0.04)
    cbar.ax.set_yticklabels(["Pruned", "Alive"])

    fig.suptitle(
        f"{payload['model_name']} | {payload['dataset']} | {payload['mode']} "
        f"({len(payload['history']) - 1} pruning iters)",
        fontsize=12,
    )
    fig.tight_layout()
    plt.show()


state = {"payload": None}
mode_options = list(MODES.keys())

model_dropdown = widgets.Dropdown(
    options=list(MODEL_NAMES),
    value=MODEL_NAMES[0],
    description="Model:",
    style={"description_width": "initial"},
)
dataset_dropdown = widgets.Dropdown(
    options=list(DATASETS),
    value=DATASETS[0],
    description="Dataset:",
    style={"description_width": "initial"},
)
mode_dropdown = widgets.Dropdown(
    options=mode_options,
    value=mode_options[0],
    description="Mode:",
    style={"description_width": "initial"},
)

iteration_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Iteration:",
    continuous_update=False,
    style={"description_width": "initial"},
)
status = widgets.HTML()
plot_output = widgets.Output()


def _render_plot(*_):
    payload = state.get("payload")
    if payload is None:
        return

    iteration_idx = min(iteration_slider.value, len(payload["history"]) - 1)
    with plot_output:
        plot_output.clear_output(wait=True)
        plot_iteration(payload, iteration_idx)


def _reload_data(*_):
    model_name = model_dropdown.value
    dataset = dataset_dropdown.value
    mode = mode_dropdown.value

    try:
        payload = load_iteration_view_data(model_name, dataset, mode)
    except Exception as exc:
        state["payload"] = None
        status.value = f"<b>Error:</b> {type(exc).__name__}: {exc}"
        with plot_output:
            plot_output.clear_output(wait=True)
        return

    state["payload"] = payload
    iteration_slider.max = len(payload["history"]) - 1
    iteration_slider.value = min(iteration_slider.value, iteration_slider.max)
    status.value = f"<b>Path:</b> {payload['json_path']}"
    _render_plot()


for control in (model_dropdown, dataset_dropdown, mode_dropdown):
    control.observe(_reload_data, names="value")
iteration_slider.observe(_render_plot, names="value")

display(
    widgets.VBox([
        widgets.HBox([model_dropdown, dataset_dropdown, mode_dropdown]),
        iteration_slider,
        status,
        plot_output,
    ])
)

_reload_data()